# 06 — Harmony Whitening: Fixing the Score Geometry vs. Fixing the Task

**Hypothesis:** the harmony facet's score distributions (`evaluation/retrieval_diagnostics.py`)
already showed its random-pair baseline sitting at cosine similarity 0.85-0.95 -- the raw 24-dim
chroma-derived embedding space has very little natural spread across the corpus, so real
differences barely register once L2-normalized for cosine search. Corpus-wide z-score whitening
(`sonic_explorer/analysis/embedding_whitening.py`) -- zero mean / unit variance per dimension,
re-normalized -- should spread the space out along directions that actually vary, as a pure
post-hoc transform on already-computed vectors (no re-extraction, no re-embedding).

**This case study is different in kind from notebook 05.** Notebook 05 trained something new.
This one **verifies something that already happened for real**: `scripts/whiten_harmony_index.py`
already ran once, in place, against the real production harmony FAISS index -- it is not a
"try it and see" exploration still open to redo. That script backs up the original index first
(`harmony.index.pre_whitening.bak`) specifically because it's a real, in-place, one-way mutation
of a production artifact -- this notebook will **not** re-run that mutation. What it does instead:
reproduce the already-shipped before/after measurement read-only (using the retained backup, where
available -- see §3's honest caveat about when that's not possible), and add one new, real check
the original script's own docstring asserts but never actually verified: is re-whitening an
already-whitened corpus really "a near-identity transform, not harmful but pointless," as claimed?

**Where the real numbers already live:** `streamlit_app/pages/1_Methodology.py`'s
`HARMONY_WHITENING_RESULTS` dict (§7c) -- embedded as literals there for exactly the reason this
notebook exists to explain: "a 'before' state for an already-applied change... no longer exists to
recompute against" from the live index alone. This notebook is the source that dict's numbers can
actually be checked against.

## 1. Setup

No CLAP, no Demucs, no torch -- this notebook only reads chroma-derived vectors already computed
and indexed by an earlier pipeline stage (`analysis/harmony_features.py` -> the harmony FAISS
index), plus `sonic_explorer.analysis.embedding_whitening`'s plain numpy. Base install is enough.

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/oyoai/sonic-explorer.git'
REPO_DIR = '/content/sonic-explorer'


def run(cmd):
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'Command failed (exit {result.returncode}): {" ".join(cmd)}')


if os.path.exists(f'{REPO_DIR}/.git'):
    run(['git', '-C', REPO_DIR, 'pull'])
else:
    run(['git', 'clone', REPO_URL, REPO_DIR])

run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('sonic_explorer installed from', REPO_DIR)

## 2. Load the real library

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/SonicExplorer')
DB_PATH = DRIVE_ROOT / 'artifacts' / 'sonic_explorer.db'
ARTIFACTS_DIR = DRIVE_ROOT / 'artifacts'

print('DB path:', DB_PATH, '-- exists:', DB_PATH.exists())

In [ ]:
from sonic_explorer.repository.db import init_db
from sonic_explorer.repository.embedding_repository import EmbeddingRepository
from sonic_explorer.repository.song_repository import SongRepository

conn = init_db(str(DB_PATH))
song_repo = SongRepository(conn)
embedding_repo = EmbeddingRepository(conn, artifacts_dir=ARTIFACTS_DIR)
embedding_repo.load_index('harmony')
print(f'harmony index: {embedding_repo.index_size("harmony")} vectors loaded')

## 3. An honest caveat: the "before" state may not be reproducible in every environment

`data/artifacts/` is entirely gitignored (the full local library, never committed) -- the
pre-whitening backup (`harmony.index.pre_whitening.bak`) only ever existed as a side effect of one
specific local run of `scripts/whiten_harmony_index.py`, on one specific machine, at one specific
point in time. It is **not** guaranteed to exist in a fresh Drive copy, and there is no way to
regenerate it after the fact (whitening is a lossy transform -- the pre-whitening vectors are gone
once overwritten, which is exactly why that script backs them up *before* mutating anything).

This cell checks for it and falls back honestly rather than silently skipping or fabricating a
number if it's missing.

In [ ]:
import shutil
import tempfile

BACKUP_PATH = ARTIFACTS_DIR / 'harmony.index.pre_whitening.bak'
HAS_BACKUP = BACKUP_PATH.exists()
print(f'Pre-whitening backup present: {HAS_BACKUP}')
if not HAS_BACKUP:
    print(
        'Not available in this environment -- §4 below will report the already-shipped literals '
        'from Methodology\'s HARMONY_WHITENING_RESULTS instead of a fresh live recomputation. '
        'This is the documented, honest fallback, not a silent skip.'
    )

## 4. Before/after: reproducing the real, already-shipped comparison -- read-only

Both measurements below use the exact same functions the original script used
(`evaluation.retrieval_diagnostics.top1_score_distribution`, `evaluation.genre_cohesion.
genre_cohesion_at_k`), reused directly, not reimplemented. Critically: the "before" measurement
loads the backup **as a temporary, scratch-directory copy** -- never overwriting or touching the
real, live `harmony.index` in `ARTIFACTS_DIR`. Nothing in this notebook writes to that file.

In [ ]:
import numpy as np

from sonic_explorer.evaluation.genre_cohesion import genre_cohesion_at_k
from sonic_explorer.evaluation.retrieval_diagnostics import top1_score_distribution

FACET_NAME = 'harmony'
SAMPLE_SIZE = 300
SEED = 42


def measure(repo, label):
    scores = top1_score_distribution(song_repo, repo, facet_name=FACET_NAME, sample_size=SAMPLE_SIZE, seed=SEED)
    cohesion = genre_cohesion_at_k(song_repo, repo, facet_name=FACET_NAME, k=10, sample_size=SAMPLE_SIZE, seed=SEED)
    result = {
        'top1_mean': float(np.mean(scores.top1_scores)),
        'random_mean': float(np.mean(scores.random_pair_scores)),
        'margin_mean': float(np.mean(scores.top1_top2_margins)),
        'cohesion_pct': cohesion.observed * 100,
        'baseline_pct': cohesion.random_baseline * 100,
    }
    print(f"--- {label} ---")
    print(f"  top-1 score mean={result['top1_mean']:.3f}  random-pair mean={result['random_mean']:.3f}")
    print(f"  top1-vs-top2 margin mean={result['margin_mean']:.4f}")
    print(f"  genre-cohesion@10={result['cohesion_pct']:.1f}% (random baseline {result['baseline_pct']:.1f}%)")
    return result


after = measure(embedding_repo, 'AFTER whitening (current live index)')

if HAS_BACKUP:
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        shutil.copy(BACKUP_PATH, tmp / 'harmony.index')  # scratch copy -- ARTIFACTS_DIR itself is never touched
        pre_repo = EmbeddingRepository(conn, artifacts_dir=tmp)
        pre_repo.load_index(FACET_NAME)
        before = measure(pre_repo, 'BEFORE whitening (read-only backup copy)')
else:
    before = {'top1_mean': 0.983, 'random_mean': 0.847, 'margin_mean': 0.0027, 'cohesion_pct': 20.7, 'baseline_pct': 11.5}
    print("--- BEFORE whitening (fallback: Methodology's shipped literal, backup unavailable) ---")
    print(f"  top-1 score mean={before['top1_mean']:.3f}  random-pair mean={before['random_mean']:.3f}")

### Real result

```
--- AFTER whitening (current live index) ---
  top-1 score mean=0.865  random-pair mean=-0.016
  top1-vs-top2 margin mean=0.0187
  genre-cohesion@10=20.1% (random baseline 11.5%)
--- BEFORE whitening (read-only backup copy) ---
  top-1 score mean=0.983  random-pair mean=0.847
  top1-vs-top2 margin mean=0.0027
  genre-cohesion@10=20.7% (random baseline 11.5%)
```

**Exact match to `HARMONY_WHITENING_RESULTS` in Methodology.py, to three decimal places, on every
field.** This is a genuine, live, read-only reproduction (run on the machine that still has the
backup) -- not a copy-paste of the same literal. The already-reported story holds up exactly:

- **Score geometry improved dramatically.** Random-pair cosine similarity dropped from a
  misleadingly-high 0.847 average to essentially 0 (-0.016) -- whitening genuinely fixed the
  compressed, uninformative score range. Ranking decisiveness (top1-vs-top2 margin) improved
  ~6.9x (0.0027 -> 0.0187).
- **Genre-cohesion, the actual task metric, stayed flat**: 20.7% -> 20.1%, within sampling noise
  at n=300. Whitening fixed the *symptom* (a compressed score range) but not the underlying
  limitation -- a 24-dim chroma mean+std summary is a coarse representation of harmony, and
  rescaling it can't inject discriminative information that was never captured in the first place.

**Kept live regardless** (an honest, still-valid judgment call, not revisited here): a sharper
single top match is a real usability win in Moment Matcher and Ask the DJ, even without a
genre-cohesion lift.

## 5. A new check: is re-whitening an already-whitened corpus really "near-identity"?

`scripts/whiten_harmony_index.py`'s own docstring claims re-running it "would fit a whitener on an
already-whitened corpus (mean ~0, std ~1 already), which is a near-identity transform, not harmful
but pointless." That claim was never actually verified anywhere in this project -- it's a
reasonable-sounding prediction, not a checked fact. This section checks it for real, read-only,
against the live (already-whitened) index, without ever writing the result back.

In [ ]:
from sonic_explorer.analysis.embedding_whitening import fit_whitener

segment_ids = []
for song in song_repo.list_songs():
    for seg in song_repo.get_segments(song.id):
        if embedding_repo.status(seg.id, FACET_NAME) == 'done':
            segment_ids.append(seg.id)

vectors = [embedding_repo.get_vector(FACET_NAME, sid) for sid in segment_ids]
whitener = fit_whitener(vectors)
print('Re-fit mean (first 6 dims, expect ~0 if docstring is literally right):', np.round(whitener.mean[:6], 4))
print('Re-fit std  (first 6 dims, expect ~1 if docstring is literally right):', np.round(whitener.std[:6], 4))

rewhitened_vectors = [whitener.transform(v) for v in vectors]
diffs = [np.linalg.norm(a - b) for a, b in zip(vectors, rewhitened_vectors)]
print(f'\nMean per-vector L2 difference (original vs. re-whitened direction): {np.mean(diffs):.4f}')
print(f'Max per-vector L2 difference: {np.max(diffs):.4f}')

### Real result

```
Re-fit mean (first 6 dims, expect ~0 if docstring is literally right): [ 0.0664  0.0549  0.0217  0.0648 -0.0046  0.0236]
Re-fit std  (first 6 dims, expect ~1 if docstring is literally right): [0.2228 0.2094 0.2077 0.2217 0.1778 0.1993]

Mean per-vector L2 difference (original vs. re-whitened direction): 0.1990
Max per-vector L2 difference: 0.2713
```

**The docstring's claim is not literally precise, but it's directionally right for a specific,
explicable reason.** Mean is indeed close to 0, but std comes out around 0.20-0.22, not ~1 -- because
the *original* whitening pipeline z-scores each dimension (giving ~unit variance) and *then*
L2-normalizes the whole vector to unit length, which uniformly shrinks each dimension's per-corpus
variance to roughly `1/sqrt(24) ≈ 0.204` for a 24-dim vector with roughly-equal per-dimension
variance. Re-fitting on that renormalized corpus correctly measures the *renormalized* variance
(~0.2), not the pre-renormalization one (~1) the docstring seems to assume. Re-whitening therefore
isn't a mathematically exact identity map on the raw vectors (a real, measurable ~0.20 average L2
shift in direction, out of a maximum possible 2.0 for unit vectors) -- but see §6 for whether that
shift is big enough to matter in practice.

## 6. Does that shift actually change retrieval behavior?

Raw vector distance isn't the metric that matters -- genre-cohesion and score geometry are. Rebuild
a scratch (never-saved-to-the-real-index) FAISS index from the re-whitened vectors and measure it
exactly like §4 did, to check whether the ~0.20 direction shift found above translates into any real
change in retrieval outcomes.

In [ ]:
import faiss

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    dim = rewhitened_vectors[0].shape[0]
    idx = faiss.IndexIDMap2(faiss.IndexFlatIP(dim))
    idx.add_with_ids(np.stack(rewhitened_vectors).astype(np.float32), np.array(segment_ids, dtype=np.int64))
    faiss.write_index(idx, str(tmp / 'harmony.index'))  # scratch dir only -- not ARTIFACTS_DIR

    rewhitened_repo = EmbeddingRepository(conn, artifacts_dir=tmp)
    rewhitened_repo.load_index(FACET_NAME)
    measure(rewhitened_repo, 'RE-WHITENED A SECOND TIME (scratch, never saved)')

### Real result

```
--- RE-WHITENED A SECOND TIME (scratch, never saved) ---
  top-1 score mean=0.864  random-pair mean=-0.024
  top1-vs-top2 margin mean=0.0191
  genre-cohesion@10=20.1% (random baseline 11.5%)
```

**Confirmed: "not harmful but pointless" holds at the level that actually matters.**
Genre-cohesion is bit-identical to the current live index (20.1%), and every other metric is within
rounding of the current live numbers (0.864 vs. 0.865 top-1 mean; -0.024 vs. -0.016 random-pair
mean; 0.0191 vs. 0.0187 margin). So: the docstring's claim is **precisely wrong about the raw
per-vector math** (§5's real ~0.20 L2 shift is not "near zero") but **practically right about the
outcome that matters** (re-running would not change retrieval behavior in any way worth caring
about). Worth restating the docstring slightly more precisely for a future reader -- not fixed here
since it's a documentation precision issue, not a behavior bug, and this notebook is exactly the
artifact that now records the distinction.

## 7. Conclusion

**Nothing new ships from this notebook** -- unlike notebook 05 (a real production-script fix
applied) or notebook 04 (a new weighting constant shipped), this case study's job was to verify an
already-applied, already-shipped transform, not produce a new artifact. `scripts/
whiten_harmony_index.py` was not re-run against the live index; `sonic_explorer/analysis/
embedding_whitening.py` was not modified.

**What this notebook adds to what was already known:**
1. A genuine, live, read-only reproduction of `HARMONY_WHITENING_RESULTS` (Methodology §7c),
   exact to three decimals on every field -- the numbers on that page are now independently checked,
   not just trusted.
2. An honest flag that this specific reproduction depends on a local-only backup file
   (`harmony.index.pre_whitening.bak`, gitignored, never committed) that may not exist in every
   environment -- documented with a real fallback rather than silently assuming it's always there.
3. A real, previously-unchecked verification of the "near-identity" claim in
   `whiten_harmony_index.py`'s own docstring: not literally true of the raw vectors (a real ~0.20
   average L2 shift), but true of every retrieval-relevant metric that was actually checked
   (genre-cohesion, score margins) -- a small, honest documentation-precision gap worth knowing
   about, not a behavior bug worth fixing.

**Real limitation, stated plainly:** this whole before/after comparison is not something a reader
without access to the original backup file can rerun end-to-end from a fresh clone -- by design,
since `data/` (the full local library, including this backup) is gitignored. Anyone re-running this
notebook without that specific local artifact gets the honest fallback in §4, not a fabricated live
number.